# Семинар 2. От промптов к агенту: messages, контекст, tools и цикл действий

**Дата:** 30 июня   
**Тема по гугл презентации:** Базовый питон, без которого не поговорить с моделью: переменные, словари, функции. Учимся читать ответ модели и доставать из него нужное.    
**Тема реальная:** как перейти от одного большого prompt к `messages`, а потом собрать простого агента с tools, state и циклом действий.

TODO - поправить тему. 


---

## 0. Повторение прошлого семинара

В прошлый раз мы научились:

- отправлять запросы к GigaChat;
- писать функцию `ask_model`;
- собирать один большой текстовый prompt;
- делать мини-чат на обычной истории текста;
- использовать модель как классификатор;
- понимать, что вызовы LLM в цикле или с последующим if/else — это ещё не агент.

Главная мысль из лецкии про промпты и как она помогает улучшить наш чат-бот:

> Сначала у нас был один большой промпт-текст.
> Сейчас мы увидим, что chat API удобнее строить через `messages`.


## 1. План семинара

Сегодня мы сначала разберём промпты, роли сообщений и формат ответа, а уже потом перейдём к tools и агентскому циклу.

К концу семинара мы должны понять:

1. чем `system`, `user` и `assistant` отличаются друг от друга;
2. как переписать большой prompt в `messages`;
3. как system prompt меняет поведение модели;
4. как задавать формат ответа и использовать few-shot-примеры;
5. почему модель иногда выдумывает и как помогает контекст;
6. что такое context engineering;
7. как tools, state и agent loop собираются в одну систему.

Главная идея:

> Хороший агент — это не просто LLM. Это LLM + правильно собранный контекст + tools + state + цикл действий.


In [92]:
import uuid
import json
import requests

import warnings
from urllib3.exceptions import InsecureRequestWarning 
warnings.filterwarnings('ignore', category=InsecureRequestWarning)

AUTH_URL = 'https://ngw.devices.sberbank.ru:9443/api/v2/oauth'
BASE_URL = 'https://gigachat.devices.sberbank.ru/api/v1/chat/completions'
MODEL_NAME = 'GigaChat-2-Pro'

AUTH_KEY = input('Вставьте ключ авторизации GigaChat и нажмите Enter: ')

print('Настройки сохранены.')
print('MODEL_NAME:', MODEL_NAME)

Настройки сохранены.
MODEL_NAME: GigaChat-2-Pro


## 2. От большого текстового промпта к chat messages - повторение лекции

В первом семинаре мы собирали один большой текст:

- история диалога;
- новый вопрос;
- инструкция, как отвечать.

Это работает. Но в chat API есть более удобная форма: список сообщений с ролями.

```python
messages = [
    {"role": "system", "content": "Ты помощник на семинаре."},
    {"role": "user", "content": "Что такое API?"},
    {"role": "assistant", "content": "API — это способ..."},
    {"role": "user", "content": "Объясни на примере доставки еды."},
]
```

`system` задаёт общую инструкцию. `user` — это сообщение пользователя. `assistant` — это предыдущий ответ модели.


## 3. System prompt

System prompt — это инструкция, которая задаёт поведение модели.

Например:

- отвечай коротко;
- объясняй как преподаватель;
- не выдумывай факты;
- если не знаешь, скажи, что не знаешь;
- возвращай ответ в заданном формате.

System prompt не является магической защитой, но сильно влияет на стиль и формат ответа.


In [204]:
def get_access_token(auth_key):
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
        'Accept': 'application/json',
        'RqUID': str(uuid.uuid4()),
        'Authorization': f'Basic {auth_key}',
    }

    data = {'scope': 'GIGACHAT_API_PERS'}

    response = requests.post(
        AUTH_URL,
        headers=headers,
        data=data,
        verify=False,
    )

    print('Статус ответа:', response.status_code)

    if response.status_code != 200:
        print('Ошибка при получении access token:')
        print(response.text)
        return None

    return response.json()['access_token']


ACCESS_TOKEN = get_access_token(AUTH_KEY)

if ACCESS_TOKEN:
    print('Access token получен.')
else:
    print('Access token не получен.')

Статус ответа: 200
Access token получен.


In [94]:
def ask_chat(messages, temperature=0.7, max_tokens=800):
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    headers = {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }

    response = requests.post(
        BASE_URL,
        headers=headers,
        json=payload,
        verify=False,
    )

    if response.status_code != 200:
        print("Ошибка запроса")
        print("Статус:", response.status_code)
        print(response.text)

        if response.status_code == 401:
            print()
            print("Возможно, истёк ACCESS_TOKEN. Перезапустите ячейку получения токена.")

        return None

    data = response.json()
    return data["choices"][0]["message"]["content"]


# ask_model оставляем для совместимости с ранними примерами.
def ask_model(prompt, temperature=0.7, max_tokens=800):
    return ask_chat([{"role": "user", "content": prompt}], temperature=temperature, max_tokens=max_tokens)


In [95]:
messages = [
    {
        "role": "system",
        "content": "Ты преподаватель программирования. Объясняй коротко, понятно и на примерах."
    },
    {
        "role": "user",
        "content": "Что такое агент?"
    }
]

answer = ask_chat(messages)
print(answer)

Агент - это программная сущность, которая автономно выполняет задачи или действия от имени пользователя или другой системы.

Пример: 
Представьте себе виртуального помощника (например, Siri или Alexa), который принимает голосовые команды, интерпретирует их и выполняет соответствующие действия (поиск информации в интернете, управление устройствами умного дома). Агент действует самостоятельно, но в рамках заданных правил и инструкций.


In [96]:
prompt = """
Ты преподаватель программирования. Объясняй коротко, понятно и на примерах.

Что такое аент?
"""

answer = ask_model(prompt)
print(answer)

### Агент — это самостоятельный объект, выполняющий определённые задачи независимо от других объектов.

#### Пример агента в Python (простая реализация):
```python
class Agent:
    def __init__(self, name):
        self.name = name

    def work(self):
        print(f"{self.name}: Выполняю свою работу...")

# Создаём агентов
agent1 = Agent("Робот-уборщик")
agent2 = Agent("Брокер акций")

# Запускаем выполнение работы агентами
agent1.work() # Робот-уборщик: Выполняю свою работу...
agent2.work() # Брокер акций: Выполняю свою работу...
```

#### Примеры агентов в реальной жизни:
- **Робот-пылесос** — самостоятельно перемещается по квартире, определяет препятствия и чистит пол.
- **Чат-бот** — общается с пользователями, обрабатывая запросы и отвечая на вопросы.
- **Финансовый аналитик** — собирает данные, прогнозирует изменения рынка и принимает решения.

Агент обладает независимостью, автономностью и способен принимать решения внутри своей области ответственности.


## 3. Как сделать агента и зачем ему tools = инструменты?

Модель может рассуждать, объяснять и писать текст. Но иногда нужно выполнять реальные действия - например, узнать актуальный прогноз погоды. 

В нашем примере у нас есть словарик (= маленькая база данных с возрастами): ключ это имя, значение это возраст. 

In [97]:
AGES = {
    'Дима': 23,
    'Маша': 17,
    'Саша': 19,
    'Аня': 16,
    'Петя': 18,
}

AGES

{'Дима': 23, 'Маша': 17, 'Саша': 19, 'Аня': 16, 'Петя': 18}

Если спросить модель: «Сколько лет Диме?», она может угадать, однако, скорее всего, скажет, что **ей неизвестно**. Причина - Но правильный ответ находится не в модели, а в нашем Python-словаре.

In [98]:
messages = [
    {
        "role": "system",
        "content": "Ты преподаватель ассистент в паспортном столе. Твоя задача - говорить сколько лет человеку по его имени"
    },
    {
        "role": "user",
        "content": "Сколько лет Диме?"
    }
]

answer = ask_chat(messages)
print(answer)

Извини, но я не могу определить возраст человека только по имени. Если у тебя есть дата рождения Димы, я смогу сказать его точный возраст.


Чтобы получить правильные ответ, у нас есть 2 способа. Первый - добавить эти данные в промпт, логичнее в систем промпт:

In [99]:
messages = [
    {
        "role": "system",
        "content": """
        Ты преподаватель ассистент в паспортном столе. Твоя задача - говорить сколько лет человеку по его имени
        
        Данные: {'Дима': 23, 'Маша': 17, 'Саша': 19, 'Аня': 16, 'Петя': 18}
        """
    },
    {
        "role": "user",
        "content": "Сколько лет Диме?"
    }
]

answer = ask_chat(messages)
print(answer)

Диме 23 года.


Проблема этого варианта - **масштабирование**. Что будет, если у нас 1млн людей (у всех имена уникальны) -> огромный систем промпт.

- Это дорого
- Даже если все это влезит в контекст (= поместится в один запрос), модель может начать путаться

**Решение**: 
- написать наш собственный API, чтобы модель могла обращаться при необходимости к этим данным. 
- Для начала напишем обучную **функцию**, а когда научим ллм ей пользоваться, она будет называться **инструмент** (на английском tool), а наши вызовы llm + уменение пользоваться этим инструментом сделают нам маленького полноценного **агента**. 

In [100]:
def get_age_raw(name: str):
    if name not in AGES:
        return {
            'ok': False,
            'error': f'Имя {name} не найдено',
        }

    return {
        'ok': True,
        'name': name,
        'age': AGES[name],
    }


print(get_age_raw('Дима'))
print(get_age_raw('Дмитрий')) # проблема, которую будем решать позже

{'ok': True, 'name': 'Дима', 'age': 23}
{'ok': False, 'error': 'Имя Дмитрий не найдено'}


Главная идея tools:

> Модель не знает данные.  
> Python знает данные.  
> Агент соединяет модель и Python.

## 4. Минимальный Python для агента - вовремя :)

Для простого агента нам понадобятся несколько конструкций.

### `dict`

Словари нужны для:

- базы данных;
- сообщений;
- результатов tools;
- JSON-действий;
- состояния агента.

In [101]:
person = {
    'name': 'Дима',
    'age': 23,
}

message = {
    'role': 'user',
    'content': 'Сколько лет Диме?',
}

print(person)
print(message)

{'name': 'Дима', 'age': 23}
{'role': 'user', 'content': 'Сколько лет Диме?'}


### и `list` тоже

In [102]:
messages = [
    {
        "role": "system",
        "content": "Ты преподаватель ассистент в паспортном столе. Твоя задача - говорить сколько лет человеку по его имени"
    },
    {
        "role": "user",
        "content": "Сколько лет Диме?"
    }
]

print(messages)

[{'role': 'system', 'content': 'Ты преподаватель ассистент в паспортном столе. Твоя задача - говорить сколько лет человеку по его имени'}, {'role': 'user', 'content': 'Сколько лет Диме?'}]


### `dataclass`

Почти всегда **данные удобнее хранить не в словаре, а в классе**. Для этих целей в питоне есть dataclass. 

Например, наш будущий агент будет делать несколько шагов.

Ему нужно хранить (= помнить):

- изначальный вопрос пользователя;
- наблюдения от вызова функции get_age_raw;
- номер текущего шага (чтобы остановить после шага N и случайно не учти в бесконечный цикл).

Это называется `state`.

In [103]:
from dataclasses import dataclass, field


@dataclass
class AgentState:
    user_question: str
    history: list[dict] = field(default_factory=list)
    steps: int = 0
    final_answer: str | None = None


state = AgentState(user_question='Сколько лет Диме?')
state.history.append({'tool_call': 'get_age', 'name': 'Дима', 'answer': get_age('Дима')})
state.steps += 1

state


AgentState(user_question='Сколько лет Диме?', history=[{'tool_call': 'get_age', 'name': 'Дима', 'answer': {'ok': True, 'name': 'Дима', 'age': 23}}], steps=1, final_answer=None)

## 5. Pydantic: проверяем данные

`Dataclass` помогает описывать структуру для человека и редактора.

Но если нужно реально проверить данные во время выполнения программы, удобно использовать Pydantic.

Начнём с простой проверки возраста.

In [104]:
from pydantic import BaseModel, Field, ValidationError


class PersonAge(BaseModel):
    name: str
    age: int = Field(ge=0, le=130)


good_person = PersonAge(name='Дима', age=23)
print(good_person)

name='Дима' age=23


In [105]:
try:
    strange_person = PersonAge(name='Гэндальф', age=2000)
    print(strange_person)
except ValidationError as error:
    print('Ошибка валидации:')
    print(error)

Ошибка валидации:
1 validation error for PersonAge
age
  Input should be less than or equal to 130 [type=less_than_equal, input_value=2000, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


Теперь улучшим tool `get_age`.

Он будет не просто возвращать возраст, а проверять, что возраст выглядит реалистично.

In [106]:
AGES = {
    'Дима': 23,
    'Маша': 17,
    'Саша': 19,
    'Аня': 16,
    'Петя': 18,

    'Гэндальф': 2000,
}


def get_age(name: str) -> dict:
    if name not in AGES:
        return {
            'ok': False,
            'error': f'Имя {name} не найдено',
        }

    raw_age = AGES[name]

    try:
        person = PersonAge(name=name, age=raw_age)
    except ValidationError:
        return {
            'ok': False,
            'error': 'Возраст найден, но не прошёл проверку',
            'name': name,
            'raw_age': raw_age,
        }

    return {
        'ok': True,
        'name': person.name,
        'age': person.age,
    }


print(get_age('Дима'))
print(get_age('Дмитрий'))
print(get_age('Гэндальф'))

{'ok': True, 'name': 'Дима', 'age': 23}
{'ok': False, 'error': 'Имя Дмитрий не найдено'}
{'ok': False, 'error': 'Возраст найден, но не прошёл проверку', 'name': 'Гэндальф', 'raw_age': 2000}


Это важный инженерный момент:

> Результат вызова инструмента (= tool) тоже нужно проверять.  
> Если tool вернул странные данные, агент не должен выдавать их как нормальный ответ.

## 6. Первый базовый агент

Теперь соберём самого простого агента.

Пока у него есть одна tool - `get_age`. Позже добавим ещё одну.

Идея очень простая:

- модель либо сразу отвечает текстом;
- либо пишет `get_age, имя`;
- Python вызывает функцию;
- если функция вызвана, результат возвращается в историю и цикл идёт дальше;
- если функции нет, мы просто возвращаем ответ модели и выходим.


In [128]:
SYSTEM_PROMPT = """Ты простой агент, который помогает узнавать возраст людей.

Если хочешь вызвать функцию, напиши так:
get_age, имя

Если функция уже не нужна, просто ответь обычным текстом.
Не добавляй ничего лишнего."""


def parse_simple_action(text: str):
    text = text.strip()

    if 'get_age' not in text:
        return {
            'type': 'final',
            'answer': text,
        }

    tool_name, arguments_text = text.split(',', 1)
    tool_name = tool_name.strip()
    name = arguments_text.strip()

    return {
        'type': 'tool_call',
        'tool_name': tool_name,
        'name': name,
    }


print(parse_simple_action('get_age, Дима'))
print(parse_simple_action('Диме 23 года'))


{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Дима'}
{'type': 'final', 'answer': 'Диме 23 года'}


In [121]:
print(SYSTEM_PROMPT)

Ты простой агент, который помогает узнавать возраст людей.

Если хочешь вызвать функцию, напиши так:
get_age, имя

Если функция уже не нужна, просто ответь обычным текстом.
Не добавляй ничего лишнего.


In [122]:
def build_simple_agent_messages(state: AgentState, system_prompt: str):
    history_text = "\n".join(
        f"{item['tool_call']}: {item['name']} -> {item['answer']}"
        for item in state.history
    )

    user_prompt = f"Вопрос пользователя: {state.user_question}\n\nИстория шагов:\n{history_text or 'пока пусто'}\n\nСейчас ответь следующим сообщением."

    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]


state = AgentState(user_question='Сколько лет Диме?')
for i, m in enumerate(build_simple_agent_messages(state, SYSTEM_PROMPT)):
    print(f"{i+1})")
    print(m['role'] + "\n" + m['content'] + "\n")


1)
system
Ты простой агент, который помогает узнавать возраст людей.

Если хочешь вызвать функцию, напиши так:
get_age, имя

Если функция уже не нужна, просто ответь обычным текстом.
Не добавляй ничего лишнего.

2)
user
Вопрос пользователя: Сколько лет Диме?

История шагов:
пока пусто

Сейчас ответь следующим сообщением.



### build_simple_agent_messages

Сначала соберём сообщение для модели, а потом запустим весь цикл.


In [123]:
def run_simple_agent(state: AgentState, system_prompt: str, max_steps: int = 4):
    for step in range(1, max_steps + 1):
        state.steps = step

        print('=' * 80)
        print(f'Шаг {step}')

        messages = build_simple_agent_messages(state, system_prompt)
        raw_answer = ask_chat(messages, temperature=0.0, max_tokens=200)

        print('Сырой ответ модели:')
        print(raw_answer)

        action = parse_simple_action(raw_answer)
        print('Действие:')
        print(action)

        if action['type'] == 'final':
            state.final_answer = action['answer']
            return action['answer']

        if action['type'] == 'tool_call':
            if action['tool_name'] == 'get_age':
                result = get_age(action['name'])
            else:
                result = f"Инструмент с названием '{action['tool_name']}' не обнаружен"

            print('Result:')
            print(result)

            state.history.append({
                'tool_call': action['tool_name'],
                'name': action['name'],
                'answer': result,
            })

    response = 'Агент остановился: достигнут лимит шагов.'
    state.final_answer = response
    return response


state = AgentState(user_question='Сколько лет Диме?')
answer = run_simple_agent(state, SYSTEM_PROMPT)

print()
print('Ответ:', answer)


Шаг 1
Сырой ответ модели:
get_age, Дима
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Дима'}
Result:
{'ok': True, 'name': 'Дима', 'age': 23}
Шаг 2
Сырой ответ модели:
Диме 23 года.
Действие:
{'type': 'final', 'answer': 'Диме 23 года.'}

Ответ: Диме 23 года.


In [124]:
state

AgentState(user_question='Сколько лет Диме?', history=[{'tool_call': 'get_age', 'name': 'Дима', 'answer': {'ok': True, 'name': 'Дима', 'age': 23}}], steps=2, final_answer='Диме 23 года.')

### 6.1 Гэндальф

Проверка не пройдена

In [125]:
state = AgentState(user_question='Сколько лет Гэндальфу?')
answer = run_simple_agent(state, SYSTEM_PROMPT)

print()
print('Ответ:', answer)

Шаг 1
Сырой ответ модели:
get_age, Гэндальф
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Гэндальф'}
Result:
{'ok': False, 'error': 'Возраст найден, но не прошёл проверку', 'name': 'Гэндальф', 'raw_age': 2000}
Шаг 2
Сырой ответ модели:
К сожалению, не могу точно определить возраст Гэндальфа. Попробуем другой источник?
Действие:
{'type': 'final', 'answer': 'К сожалению, не могу точно определить возраст Гэндальфа. Попробуем другой источник?'}

Ответ: К сожалению, не могу точно определить возраст Гэндальфа. Попробуем другой источник?


### 6.2 Дмитрий, вместо Дима - не работает

In [126]:
state = AgentState(user_question='Сколько лет Дмитрию?')
answer = run_simple_agent(state, SYSTEM_PROMPT)

print()
print('Ответ:', answer)

Шаг 1
Сырой ответ модели:
get_age, Дмитрий
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Дмитрий'}
Result:
{'ok': False, 'error': 'Имя Дмитрий не найдено'}
Шаг 2
Сырой ответ модели:
К сожалению, я не смог узнать возраст человека с именем Дмитрий. Возможно, он не указан в базе данных. Попробуете еще раз с другим именем?
Действие:
{'type': 'final', 'answer': 'К сожалению, я не смог узнать возраст человека с именем Дмитрий. Возможно, он не указан в базе данных. Попробуете еще раз с другим именем?'}

Ответ: К сожалению, я не смог узнать возраст человека с именем Дмитрий. Возможно, он не указан в базе данных. Попробуете еще раз с другим именем?


## 7. Few-shot для вариаций имени

Если имя не нашлось с первого раза, можно показать модели несколько примеров.

Тогда она будет чаще пробовать похожие имена и не застрянет на первом ответе.

Примеры простые:

- `Дмитрий` -> `Дима`, `Димон`, `Димка`;
- `Мария` -> `Маша`;
- `Анна` -> `Аня`;
- `Александр` -> `Саша`.

Ниже мы просто меняем system prompt и смотрим, как меняется поведение модели.


In [137]:
SYSTEM_PROMPT_V2 = """Ты простой агент, который помогает узнавать возраст людей.

Если хочешь вызвать функцию, напиши так:
get_age, имя

Если имя не найдено, попробуй другой вариант имени.
Примеры:
- Дмитрий -> Дима
- Дмитрий -> Димон
- Дмитрий -> Димка
- Мария -> Маша
- Александр -> Саша

Пробуй несколько вариантов по очереди. Обязательно передавай в формате:
get_age, имя

Попробуй подобрать имя макисмальное число раз. Не останавливайса на неудачах.

Если функция уже не нужна, просто ответь обычным текстом.
Не добавляй ничего лишнего."""


state = AgentState(user_question='Сколько лет Дмитрию?')
answer = run_simple_agent(state, SYSTEM_PROMPT_V2)

print()
print('Ответ:', answer)


Шаг 1
Сырой ответ модели:
get_age, Дмитрий
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Дмитрий'}
Result:
{'ok': False, 'error': 'Имя Дмитрий не найдено'}
Шаг 2
Сырой ответ модели:
get_age, Дима
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Дима'}
Result:
{'ok': True, 'name': 'Дима', 'age': 23}
Шаг 3
Сырой ответ модели:
Диме 23 года.
Действие:
{'type': 'final', 'answer': 'Диме 23 года.'}

Ответ: Диме 23 года.


### 7.1 Анна -> Аня, этого не было в промпте, это общие знания модели
В этом и замечательность llm - общие знания. Для этого ее и дообучают / переобучают. 

In [134]:
state = AgentState(user_question='Сколько лет Анне?')
answer = run_simple_agent(state, SYSTEM_PROMPT_V2)

print()
print('Ответ:', answer)

Шаг 1
Сырой ответ модели:
get_age, Анна
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Анна'}
Result:
{'ok': False, 'error': 'Имя Анна не найдено'}
Шаг 2
Сырой ответ модели:
get_age, Аня
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Аня'}
Result:
{'ok': True, 'name': 'Аня', 'age': 16}
Шаг 3
Сырой ответ модели:
Ане 16 лет.
Действие:
{'type': 'final', 'answer': 'Ане 16 лет.'}

Ответ: Ане 16 лет.


### 7.2 Зачем ограничивать число шагов

Если агент слишком долго пытается подобрать имя, нужен лимит шагов.

Иначе он может долго крутиться вокруг похожих вариантов.

Если честно, модель слишком умная и демонстрационный пример не удался. 

In [140]:
state = AgentState(user_question='Сколько лет Георгию?')
answer = run_simple_agent(state, SYSTEM_PROMPT_V2)

print()
print('Ответ:', answer)

Шаг 1
Сырой ответ модели:
get_age, Георгий
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Георгий'}
Result:
{'ok': False, 'error': 'Имя Георгий не найдено'}
Шаг 2
Сырой ответ модели:
get_age, Гоша
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Гоша'}
Result:
{'ok': False, 'error': 'Имя Гоша не найдено'}
Шаг 3
Сырой ответ модели:
get_age, Герасим
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Герасим'}
Result:
{'ok': False, 'error': 'Имя Герасим не найдено'}
Шаг 4
Сырой ответ модели:
Георгий может также называться как Гордей или Георгиос. Попробуешь эти варианты?
Действие:
{'type': 'final', 'answer': 'Георгий может также называться как Гордей или Георгиос. Попробуешь эти варианты?'}

Ответ: Георгий может также называться как Гордей или Георгиос. Попробуешь эти варианты?


In [141]:
state = AgentState(user_question='Сколько лет Георгию?')
answer = run_simple_agent(state, SYSTEM_PROMPT_V2, max_steps=3)

print()
print('Ответ:', answer)

Шаг 1
Сырой ответ модели:
get_age, Георгий
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Георгий'}
Result:
{'ok': False, 'error': 'Имя Георгий не найдено'}
Шаг 2
Сырой ответ модели:
get_age, Гоша
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Гоша'}
Result:
{'ok': False, 'error': 'Имя Гоша не найдено'}
Шаг 3
Сырой ответ модели:
get_age, Герасим
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Герасим'}
Result:
{'ok': False, 'error': 'Имя Герасим не найдено'}

Ответ: Агент остановился: достигнут лимит шагов.


### 7.3 Минутка интерактива - попробуйте свое имя и свой возраст

In [142]:
#TODO - добавьте свои данные cюда - например, вместо Миши и 12
AGES['Миша'] = 18


state = AgentState(user_question='Сколько лет Михаилу?') #TODO поправьте вопрос на свое имя или его вариацию
answer = run_simple_agent(state, SYSTEM_PROMPT_V2, max_steps=10)

print()
print('Ответ:', answer)


Шаг 1
Сырой ответ модели:
get_age, Михаил
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Михаил'}
Result:
{'ok': False, 'error': 'Имя Михаил не найдено'}
Шаг 2
Сырой ответ модели:
get_age, Миша
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Миша'}
Result:
{'ok': True, 'name': 'Миша', 'age': 18}
Шаг 3
Сырой ответ модели:
Михаилу 18 лет.
Действие:
{'type': 'final', 'answer': 'Михаилу 18 лет.'}

Ответ: Михаилу 18 лет.


In [ ]:
AGES = {
    'Дима': 23,
    'Маша': 17,
    'Саша': 19,
    'Аня': 16,
    'Петя': 18,

    'Гэндальф': 2000,
}

## 8. Новая tool: братья и сёстры

Теперь добавим ещё одну tool.

Она будет отвечать только `True` или `False`: есть ли у человека братья или сёстры.

Теперь у агента есть две похожие tools, и интересно посмотреть, как он выбирает между ними в зависимости от вопроса.


In [143]:
HAS_SIBLINGS = {
    'Дима': True,
    'Маша': False,
    'Саша': True,
    'Аня': False,
    'Петя': True,
}


def has_siblings(name: str):
    if name not in HAS_SIBLINGS:
        return {
            'ok': False,
            'error': f'Имя {name} не найдено',
        }

    return {
        'ok': True,
        'name': name,
        'has_siblings': HAS_SIBLINGS[name],
    }


print(has_siblings('Дима'))
print(has_siblings('Маша'))


{'ok': True, 'name': 'Дима', 'has_siblings': True}
{'ok': True, 'name': 'Маша', 'has_siblings': False}


Теперь можно спрашивать агента разными способами.

Например:

- `Сколько лет Диме?`;
- `Есть ли у Димы братья и сёстры?`;
- `Сколько лет Диме и есть ли у него братья?`.

Дальше будет полезно посмотреть на то, как меняется ответ, если имя есть в одной базе, но нет в другой.

Это уже не про одну единственную правильную tool, а про выбор нужного инструмента по смыслу вопроса.


In [ ]:
SYSTEM_PROMPT_V3 = """Ты простой агент, который помогает узнавать возраст людей и проверять, есть ли у них братья и сёстры.

У тебя есть две tools:
- get_age, имя -> вернуть возраст человека;
- has_siblings, имя -> вернуть True или False, есть ли у человека братья и сёстры.

Если вопрос про возраст, используй get_age.
Если вопрос про братьев и сестёр, используй has_siblings.
Если вопрос смешанный, можешь вызвать обе tools по очереди.
Обязательно используй формат: название тулы и имя через запятую. Это два слова, разделенные запятой. Например:
get_age, Дима
has_siblings, Дима

Не используй ":" как разделитель

За один раз ты можешь вызвать только одну tool.

Если имя не найдено в одной базе, попробуй другую tool, если она подходит по смыслу вопроса.
Например:
- если не нашёл возраст, но нужен только факт про братьев и сестёр, попробуй has_siblings;
- если не нашёл братьев и сестёр, но нужен возраст, попробуй get_age;
- если нужен возраст для Дмитрия, попробуй Дима;

Пробуй инструменты по очереди и не добавляй ничего лишнего."""


state = AgentState(user_question='Есть ли у Димы братья и сёстры?')
answer = run_simple_agent(state, SYSTEM_PROMPT_V3)

print()
print('Ответ:', answer)


Шаг 1
Сырой ответ модели:
has_siblings, Дима
Действие:
{'type': 'final', 'answer': 'has_siblings, Дима'}

Ответ: has_siblings, Дима


In [ ]:
def parse_simple_action_v2(text: str):
    text = text.strip()

    if 'get_age' not in text and 'has_siblings' not in text: # NEW "and 'has_siblings' not in text"
        return {
            'type': 'final',
            'answer': text,
        }

    tool_name, arguments_text = text.split(',', 1)
    tool_name = tool_name.strip()
    name = arguments_text.strip()

    return {
        'type': 'tool_call',
        'tool_name': tool_name,
        'name': name,
    }

def run_simple_agent_v2(state: AgentState, system_prompt: str, max_steps: int = 4):
    for step in range(1, max_steps + 1):
        state.steps = step

        print('=' * 80)
        print(f'Шаг {step}')

        messages = build_simple_agent_messages(state, system_prompt)
        raw_answer = ask_chat(messages, temperature=0.0, max_tokens=200)

        print('Сырой ответ модели:')
        print(raw_answer)

        action = parse_simple_action_v2(raw_answer)
        print('Действие:')
        print(action)

        if action['type'] == 'final':
            state.final_answer = action['answer']
            return action['answer']

        if action['type'] == 'tool_call':
            if action['tool_name'] == 'get_age':
                result = get_age(action['name'])
            # NEW 
            elif action['tool_name'] == 'has_siblings':
                result = has_siblings(action['name'])
            # NEW 
            else:
                result = f"Инструмент с названием '{action['tool_name']}' не обнаружен"

            print('Result:')
            print(result)

            state.history.append({
                'tool_call': action['tool_name'],
                'name': action['name'],
                'answer': result,
            })

    response = 'Агент остановился: достигнут лимит шагов.'
    state.final_answer = response
    return response

In [ ]:
state = AgentState(user_question='Сколько лет Диме и есть ли у него братья?')
answer = run_simple_agent_v2(state, SYSTEM_PROMPT_V3, max_steps=6)

print()
print('Ответ:', answer)

Шаг 1
Сырой ответ модели:
get_age, Дима
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Дима'}
Result:
{'ok': True, 'name': 'Дима', 'age': 23}
Шаг 2
Сырой ответ модели:
has_siblings, Дима
Действие:
{'type': 'tool_call', 'tool_name': 'has_siblings', 'name': 'Дима'}
Result:
{'ok': True, 'name': 'Дима', 'has_siblings': True}
Шаг 3
Сырой ответ модели:
Диме 23 года и у него есть братья или сёстры.
Действие:
{'type': 'final', 'answer': 'Диме 23 года и у него есть братья или сёстры.'}

Ответ: Диме 23 года и у него есть братья или сёстры.


In [157]:
state = AgentState(user_question='Сколько лет Дмитрию и есть ли у него братья?')
answer = run_simple_agent_v2(state, SYSTEM_PROMPT_V3, max_steps=6)

print()
print('Ответ:', answer)

Шаг 1
Сырой ответ модели:
get_age, Дмитрий
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Дмитрий'}
Result:
{'ok': False, 'error': 'Имя Дмитрий не найдено'}
Шаг 2
Сырой ответ модели:
get_age, Дима
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Дима'}
Result:
{'ok': True, 'name': 'Дима', 'age': 23}
Шаг 3
Сырой ответ модели:
has_siblings, Дима
Действие:
{'type': 'tool_call', 'tool_name': 'has_siblings', 'name': 'Дима'}
Result:
{'ok': True, 'name': 'Дима', 'has_siblings': True}
Шаг 4
Сырой ответ модели:
Диме 23 года и у него есть братья или сёстра.
Действие:
{'type': 'final', 'answer': 'Диме 23 года и у него есть братья или сёстра.'}

Ответ: Диме 23 года и у него есть братья или сёстра.


Отдельно приятно, что модель поняла Дмитрий -> Дима только один раз и вызвала две тулы с Димой, а не каждый раз пыталась подбирать имя. 

### 7.1 Вторая минутка интерактива - попробуйте добавить свои данные

In [ ]:
#TODO - добавьте свои данные cюда - например, вместо Миши
AGES['Миша'] = 18
HAS_SIBLINGS['Миша'] = False

state = AgentState(user_question='Что ты знаешь о Михаиле?') #TODO поправьте вопрос на свое имя
answer = run_simple_agent_v2(state, SYSTEM_PROMPT_V3, max_steps=10)

Шаг 1
Сырой ответ модели:
get_age, Михаил  
has_siblings, Михаил
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Михаил  \nhas_siblings, Михаил'}
Result:
{'ok': False, 'error': 'Имя Михаил  \nhas_siblings, Михаил не найдено'}
Шаг 2
Сырой ответ модели:
get_age, Миша
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Миша'}
Result:
{'ok': True, 'name': 'Миша', 'age': 18}
Шаг 3
Сырой ответ модели:
has_siblings, Миша
Действие:
{'type': 'tool_call', 'tool_name': 'has_siblings', 'name': 'Миша'}
Result:
{'ok': True, 'name': 'Миша', 'has_siblings': False}
Шаг 4
Сырой ответ модели:
Человек по имени Михаил, также известный как Миша, age 18, у него нет братьев и сестёр.
Действие:
{'type': 'final', 'answer': 'Человек по имени Михаил, также известный как Миша, age 18, у него нет братьев и сестёр.'}


In [ ]:
AGES = {
    'Дима': 23,
    'Маша': 17,
    'Саша': 19,
    'Аня': 16,
    'Петя': 18,

    'Гэндальф': 2000,
}

HAS_SIBLINGS = {
    'Дима': True,
    'Маша': False,
    'Саша': True,
    'Аня': False,
    'Петя': True,
}

## 9. Что мы сделали

Сегодня мы собрали маленького агента и посмотрели, как он ведёт себя на разных вопросах.

Главная идея такая:

- LLM сама по себе умеет отвечать текстом;
- агент появляется тогда, когда мы добавляем tools, историю шагов и цикл;
- prompt engineering - это когда мы подбираем system prompt, примеры и инструкции так, чтобы модель делала нужные шаги;
- классическое программирование - это когда мы пишем сам цикл, обработку ошибок, словари, tools и решаем, что делать, если модель ошиблась или вызвала не ту функцию.

Мы уже экспериментировали с разными вариантами промпта, добавляли few-shot-примеры и смотрели, как меняется поведение модели.

Отдельная полезная идея:

- если модель часто вызывает две tools подряд (это было, но не вошло в упрощенную версию семинара), это можно упростить в коде;
- если один шаг всегда повторяется, можно вынести его в отдельную tool;
- если что-то ломается, это тоже задача обычного Python-кода, а не только промпта.

> То есть агент - это не только LLM. Это LLM + prompt engineering + tools + обычное программирование.

## 10. Задание: поэкспериментируйте с промптами

В этой части мы не меняем код агента.

Наша цель - посмотреть, как сильно поведение агента зависит от system prompt и от формулировки вопроса.

Что нужно сделать:

- написать свой собственный `SYSTEM_PROMPT_V4`;
- для этого я уже скопировал вниз `SYSTEM_PROMPT_V3` - его можно взять за основу и немного поменять инструкции;
- запустить агента на нескольких вопросах;
- посмотреть, где он отвечает хорошо, а где начинает путаться;
- попробовать сломать агента странным вопросом;
- при желании вы можете добавлять новые имена в словарики `AGES` and `HAS_SIBLINGS`;
- можеть ограничивать максимальное число шагов через `max_steps`.

Зачем это нужно:

- чтобы увидеть, что маленькие изменения в промпте сильно влияют на ответ;
- чтобы понять, где модель следует инструкции, а где начинает импровизировать;
- чтобы научиться проверять, как агент ведёт себя в пограничных случаях.

Примеры вопросов:

- `Сколько лет Диме?`;
- `Есть ли у Димы братья и сёстры?`;
- `Сколько лет Дмитрию и есть ли у него братья?`;
- `Сколько лет Гоше?`;
- `Сколько лет Диме, если у него есть братья?`.
- `Я Дима. Что ты знаешь обо мне?`

Попробуйте менять и промпт, и сам вопрос.
Потом сравните, что именно изменилось в поведении агента.


In [ ]:
SYSTEM_PROMPT_V3 = """Ты простой агент, который помогает узнавать возраст людей и проверять, есть ли у них братья и сёстры.

У тебя есть две tools:
- get_age, имя -> вернуть возраст человека;
- has_siblings, имя -> вернуть True или False, есть ли у человека братья и сёстры.

Если вопрос про возраст, используй get_age.
Если вопрос про братьев и сестёр, используй has_siblings.
Если вопрос смешанный, можешь вызвать обе tools по очереди.
Обязательно используй формат: название тулы и имя через запятую. Это два слова, разделенные запятой. Например:
get_age, Дима
has_siblings, Дима

Не используй ":" как разделитель

За один раз ты можешь вызвать только одну tool.

Если имя не найдено в одной базе, попробуй другую tool, если она подходит по смыслу вопроса.
Например:
- если не нашёл возраст, но нужен только факт про братьев и сестёр, попробуй has_siblings;
- если не нашёл братьев и сестёр, но нужен возраст, попробуй get_age;
- если нужен возраст для Дмитрия, попробуй Дима;

Пробуй инструменты по очереди и не добавляй ничего лишнего."""

In [ ]:
SYSTEM_PROMPT_YOURS = """...""" # TODO поправьте системное сообщение

In [ ]:
#TODO - добавьте новые данные при необходимости 
# AGES['Миша'] = 18
# HAS_SIBLINGS['Миша'] = False

In [ ]:
state1 = AgentState(user_question='...') #TODO поправьте вопрос на свое имя
answer1 = run_simple_agent_v2(state1, SYSTEM_PROMPT_YOURS, max_steps=10)

In [ ]:
state2 = AgentState(user_question='...') #TODO поправьте вопрос на свое имя
answer2 = run_simple_agent_v2(state2, SYSTEM_PROMPT_YOURS, max_steps=10)

In [ ]:
state3 = AgentState(user_question='...') #TODO поправьте вопрос на свое имя
answer3 = run_simple_agent_v2(state3, SYSTEM_PROMPT_YOURS, max_steps=10)

## 11. Задание: нужна ли новая tool?

Ещё одна полезная tool - проверка, достиг ли человек совершеннолетия.

Идея простая:

- на вход tool получает имя;
- внутри она смотрит возраст;
- потом возвращает `True`, если человек совершеннолетний, и `False` в противном случае.

**Вопрос**: нужно ли писать отдельную tool или можно обойтись проще?

**Ответ**: можно проще: возраст уже есть, значит агент может сам решить, совершеннолетний человек или нет.

Уже сейчас можно попробовать:

- спросить у агента про возраст;
- спросить, совершеннолетний ли человек;
- спросить и про возраст, и про совершеннолетие в одном вопросе;
- посмотреть, хватает ли уже существующих tools, чтобы ответить правильно.

При желании можно попробовать не добавлять новую tool, а просто объяснить агенту в system prompt, как использовать уже существующие tools:

- сначала вызови `get_age`;
- потом посмотри на возраст;
- если возраст 18 или больше, скажи, что человек совершеннолетний;
- если меньше 18, скажи, что несовершеннолетний.

То есть в этом задании важно не добавить ещё один инструмент, а понять, когда tool действительно нужна, а когда задачу можно решить через хороший prompt и уже готовые tools.

### 11.1 Solution
Можно вообще ничего не делать :)

In [ ]:
state = AgentState(user_question='Является ли Дима совершеннолетним?')
answer = run_simple_agent_v2(state, SYSTEM_PROMPT_V3, max_steps=10)

Шаг 1
Сырой ответ модели:
get_age, Дима
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Дима'}
Result:
{'ok': True, 'name': 'Дима', 'age': 23}
Шаг 2
Сырой ответ модели:
Да, Дима является совершеннолетним.
Действие:
{'type': 'final', 'answer': 'Да, Дима является совершеннолетним.'}


In [ ]:
state = AgentState(user_question='Является ли Петя совершеннолетним?')
answer = run_simple_agent_v2(state, SYSTEM_PROMPT_V3, max_steps=10)

Шаг 1
Сырой ответ модели:
get_age, Петя
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Петя'}
Result:
{'ok': True, 'name': 'Петя', 'age': 18}
Шаг 2
Сырой ответ модели:
Да, Петя является совершеннолетним.
Действие:
{'type': 'final', 'answer': 'Да, Петя является совершеннолетним.'}


**Вопрос**: Почему модель знает, что 18 это совершеннолетний?

**Ответ**: Она училась на русских данных. В других странах, возраст совершеннолетия может быть другой. Например, 20 - для Новой Зеландии. Если мы хотим, чтобы модель об этом знала - нужно поправить system prompt. 

In [179]:
new_part = "Если тебя спрашивают про совершеннолетие, вызови тулу get_age и реши: если возраст больше 19, то человек является совершеннолетним. 18 и 19 это еще не совершеннолетний."

SYSTEM_PROMPT_WITHOUTH_IS_ADULT = f"""Ты простой агент, который помогает узнавать возраст людей и проверять, есть ли у них братья и сёстры.

У тебя есть две tools:
- get_age, имя -> вернуть возраст человека;
- has_siblings, имя -> вернуть True или False, есть ли у человека братья и сёстры.

Если вопрос про возраст, используй get_age.
Если вопрос про братьев и сестёр, используй has_siblings.
Если вопрос смешанный, можешь вызвать обе tools по очереди.
Обязательно используй формат: название тулы и имя через запятую. Это два слова, разделенные запятой. Например:
get_age, Дима
has_siblings, Дима

Не используй ":" как разделитель

За один раз ты можешь вызвать только одну tool.

Если имя не найдено в одной базе, попробуй другую tool, если она подходит по смыслу вопроса.
Например:
- если не нашёл возраст, но нужен только факт про братьев и сестёр, попробуй has_siblings;
- если не нашёл братьев и сестёр, но нужен возраст, попробуй get_age;
- если нужен возраст для Дмитрия, попробуй Дима;

{new_part}

Пробуй инструменты по очереди и не добавляй ничего лишнего."""

In [ ]:
state = AgentState(user_question='Является ли Петя совершеннолетним?')
answer = run_simple_agent_v2(state, SYSTEM_PROMPT_WITHOUTH_IS_ADULT, max_steps=10)

Шаг 1
Сырой ответ модели:
get_age, Петя
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Петя'}
Result:
{'ok': True, 'name': 'Петя', 'age': 18}
Шаг 2
Сырой ответ модели:
Возраст Пети равен 18 годам, что означает, что он еще не является совершеннолетним.
Действие:
{'type': 'final', 'answer': 'Возраст Пети равен 18 годам, что означает, что он еще не является совершеннолетним.'}


На крайний случай можно написать новую тулу. 

In [183]:
SYSTEM_PROMPT_WITH_IS_ADULT = """..."""
AGE_LIMIT = 20

def is_adult(name: str):
    age_info = get_age(name)
    if not age_info.get('ok'):
        return {'ok': False, 'error': age_info.get('error')}

    return {
        'ok': True,
        'name': name,
        'is_adult': age_info['age'] >= AGE_LIMIT,
    }


print(is_adult('Дима'))
print(is_adult('Дмитрий'))

{'ok': True, 'name': 'Дима', 'is_adult': True}
{'ok': False, 'error': 'Имя Дмитрий не найдено'}


И попробовать добавить агенту - поправить system prompt, run_simple_agent_v2 и parse_simple_action_v2 - но это за гранью семинара

## 12 Задание - придумать необычные вопросы, которые агент сможет решить используя уже существующие тулы

In [205]:
state = AgentState(user_question='Сколько детей у родителей Димы: один или больше?')
answer = run_simple_agent_v2(state, SYSTEM_PROMPT_V3, max_steps=10)

Шаг 1
Сырой ответ модели:
has_siblings, Дима
Действие:
{'type': 'tool_call', 'tool_name': 'has_siblings', 'name': 'Дима'}
Result:
{'ok': True, 'name': 'Дима', 'has_siblings': True}
Шаг 2
Сырой ответ модели:
У родителей Димы больше одного ребёнка, так как установлено, что у Димы есть братья и сёстрары.
Действие:
{'type': 'final', 'answer': 'У родителей Димы больше одного ребёнка, так как установлено, что у Димы есть братья и сёстрары.'}


In [206]:
state = AgentState(user_question='Является ли Дима пенсионером?')
answer = run_simple_agent_v2(state, SYSTEM_PROMPT_V3, max_steps=10)

Шаг 1
Сырой ответ модели:
get_age, Дима
Действие:
{'type': 'tool_call', 'tool_name': 'get_age', 'name': 'Дима'}
Result:
{'ok': True, 'name': 'Дима', 'age': 23}
Шаг 2
Сырой ответ модели:
Дима не является пенсионером, так как его возраст составляет 23 года.
Действие:
{'type': 'final', 'answer': 'Дима не является пенсионером, так как его возраст составляет 23 года.'}


In [ ]:
#TODO 